# Lesson 5: Leveraging Assistants API for SQL Databases

## Setup

- **Ollama / Groq / OpenAI**: jalankan bagian *Function calling universal*.
- **Azure / OpenAI resmi**: bagian *Assistants API* opsional.

In [ ]:
import json
from llm_setup import (
    check_llm_config,
    get_model_name,
    get_openai_client,
    print_provider_info,
    supports_assistants_api,
)

import Helper
from Helper import get_positive_cases_for_state_on_date
from Helper import get_hospitalized_increase_for_state_on_date
from assistant_helper import run_tool_calling_chat

issues = check_llm_config()
if issues:
    raise EnvironmentError("\n".join(issues))

print_provider_info()
client = get_openai_client()
MODEL = get_model_name()
print(f"Assistants API tersedia: {supports_assistants_api()}")

## Function calling universal (Ollama / Groq / OpenAI / Azure)

Alternatif L5 tanpa Assistants API.

In [ ]:
QUESTION = "how many hospitalized people we had in Alaska the 2021-03-05?"

available_functions = {
    "get_positive_cases_for_state_on_date": get_positive_cases_for_state_on_date,
    "get_hospitalized_increase_for_state_on_date": get_hospitalized_increase_for_state_on_date,
}

answer = run_tool_calling_chat(
    client=client,
    model=MODEL,
    user_message=QUESTION,
    tools=Helper.tools_sql,
    available_functions=available_functions,
)

print(answer)

## Assistants API (opsional — Azure / OpenAI saja)

Lewati jika memakai **Ollama** atau **Groq**.

In [ ]:
if not supports_assistants_api():
    print("Assistants API tidak dipakai untuk provider ini.")
else:
    client_az = get_openai_client(api_version="2024-02-15-preview")
    assistant = client_az.beta.assistants.create(
        instructions="You are an assistant answering questions about a Covid dataset.",
        model=MODEL,
        tools=Helper.tools_sql,
    )
    thread = client_az.beta.threads.create()
    print(thread)

In [ ]:
if supports_assistants_api():
    message = client_az.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content="how many hospitalized people we had in Alaska the 2021-03-05?",
    )
    run = client_az.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id,
    )
    print(message)
    print(run.status)

In [ ]:
import time

if supports_assistants_api():
    status = run.status
    while status not in ["completed", "cancelled", "expired", "failed"]:
        time.sleep(5)
        run = client_az.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
        status = run.status
        print(f"Status: {status}")
        if status == "requires_action":
            tool_outputs = []
            for tool_call in run.required_action.submit_tool_outputs.tool_calls:
                fn = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                fn_map = {
                    "get_positive_cases_for_state_on_date": get_positive_cases_for_state_on_date,
                    "get_hospitalized_increase_for_state_on_date": get_hospitalized_increase_for_state_on_date,
                }
                result = fn_map[fn](
                    state_abbr=args.get("state_abbr"),
                    specific_date=args.get("specific_date"),
                )
                tool_outputs.append({"tool_call_id": tool_call.id, "output": str(result)})
            run = client_az.beta.threads.runs.submit_tool_outputs(
                thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs
            )

    messages = client_az.beta.threads.messages.list(thread_id=thread.id)
    print(messages.model_dump_json(indent=2))